<a href="https://colab.research.google.com/github/birolcabukusta/Tools/blob/main/pubmed_formatter.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Pubmed Formatter**

This code takes the Pubmed-formatted list of publications and converts them to

> Title \
>List of authors (in italics) \
> Journal, year and volume \
> DOI

Input format should look like this:
> PMID- 42212855\
>OWN - NLM \
>STAT- MEDLINE \
>DCOM- 20260612 \
>LR  - 20260726 \
>IS  - 1520-6882 (Electronic) \
>IS  - 0003-2700 (Print) \
>IS  - 0003-2700 (Linking) \
>VI  - 98 \
>IP  - 22 \
>DP  - 2026 Jun 9 \
>TI  - Isotope Decluttering Reduc \

and can be obtained from such a link: `https://pubmed.ncbi.nlm.nih.gov/clipboard/?sort=date&format=pubmed` where `format=pubmed`.

> Code is generated by ChatGPT




In [ ]:
# @title

import re
import html
import ipywidgets as widgets
from IPython.display import display, HTML


def split_records(text):
    """Split pasted MEDLINE text into separate PMID records."""
    text = text.replace("\r\n", "\n").replace("\r", "\n").strip()

    records = re.split(r"(?=^PMID-\s*)", text, flags=re.MULTILINE)
    return [record.strip() for record in records if record.strip()]


def extract_field(record, field):
    """
    Extract a MEDLINE field, including continuation lines.

    Example:
    TI  - First part of title
          continued title text.
    """
    lines = record.splitlines()
    pattern = re.compile(rf"^{re.escape(field)}\s*-\s*(.*)$")

    for i, line in enumerate(lines):
        match = pattern.match(line)

        if match:
            value_parts = [match.group(1).strip()]

            for following_line in lines[i + 1:]:
                # Stop when a new MEDLINE field starts
                if re.match(r"^[A-Z0-9]{2,5}\s*-", following_line):
                    break

                # Wrapped MEDLINE lines normally begin with spaces
                if following_line.startswith("      "):
                    value_parts.append(following_line.strip())
                else:
                    break

            return re.sub(r"\s+", " ", " ".join(value_parts)).strip()

    return None


def extract_all_fields(record, field):
    """Extract all occurrences of a repeated field, such as AU."""
    pattern = re.compile(
        rf"^{re.escape(field)}\s*-\s*(.+)$",
        flags=re.MULTILINE
    )

    return [match.strip() for match in pattern.findall(record)]


def format_author(author):
    """
    Convert PubMed AU format from:
        Cabukusta B
    to:
        B Cabukusta

    Also supports compound surnames:
        van der Vliet A
    becomes:
        A van der Vliet
    """
    parts = author.strip().split()

    if len(parts) < 2:
        return author.strip()

    initials = parts[-1]
    surname = " ".join(parts[:-1])

    return f"{initials} {surname}"


def extract_doi(record):
    """Extract the first DOI marked with [doi] from LID or AID."""
    doi_patterns = [
        r"^LID\s*-\s*(10\.\S+?)\s*\[doi\]",
        r"^AID\s*-\s*(10\.\S+?)\s*\[doi\]"
    ]

    for pattern in doi_patterns:
        match = re.search(pattern, record, flags=re.MULTILINE | re.IGNORECASE)

        if match:
            return match.group(1).strip().rstrip(".")

    return None


def format_record(record):
    """Format one MEDLINE record."""
    title = extract_field(record, "TI") or "Title not available"
    journal = extract_field(record, "TA") or "Journal not available"
    publication_date = extract_field(record, "DP")
    volume = extract_field(record, "VI")
    doi = extract_doi(record)

    authors = extract_all_fields(record, "AU")
    formatted_authors = [format_author(author) for author in authors]

    if formatted_authors:
        author_text = ", ".join(formatted_authors)
    else:
        author_text = "Authors not available"

    year_match = re.search(r"\b(?:19|20)\d{2}\b", publication_date or "")
    year = year_match.group(0) if year_match else None

    publication_parts = [journal]

    if year:
        publication_parts.append(year)

    if volume:
        publication_parts.append(f"Vol. {volume}")

    publication_line = ", ".join(publication_parts)
    doi_line = f"DOI: {doi}" if doi else "DOI: not available"

    return {
        "title": title,
        "authors": author_text,
        "publication": publication_line,
        "doi": doi_line
    }


def format_medline_text(text):
    """Format every PMID record in the pasted text."""
    records = split_records(text)

    if not records:
        raise ValueError("No PMID records were found in the pasted text.")

    return [format_record(record) for record in records]


# ---------------------------
# Colab interface
# ---------------------------

input_box = widgets.Textarea(
    value="",
    placeholder=(
        "Paste one or more PubMed MEDLINE records here.\n\n"
        "The text should begin with something like:\n"
        "PMID- 42212855"
    ),
    description="",
    layout=widgets.Layout(
        width="100%",
        height="400px"
    )
)

format_button = widgets.Button(
    description="Format references",
    button_style="primary",
    icon="check"
)

clear_button = widgets.Button(
    description="Clear",
    icon="trash"
)

copy_button = widgets.Button(
    description="Copy plain text",
    icon="copy"
)

output_area = widgets.Output()

formatted_plain_text = ""


def build_plain_text(references):
    """Create plain text with Markdown italics markers."""
    blocks = []

    for reference in references:
        block = (
            f"{reference['title']}\n"
            f"*{reference['authors']}*\n"
            f"{reference['publication']}\n"
            f"{reference['doi']}"
        )
        blocks.append(block)

    return "\n\n".join(blocks)


def build_html(references):
    """Create rendered HTML output with italic authors."""
    blocks = []

    for reference in references:
        title = html.escape(reference["title"])
        authors = html.escape(reference["authors"])
        publication = html.escape(reference["publication"])
        doi = html.escape(reference["doi"])

        block = f"""
        <div style="
            margin: 0 0 24px 0;
            padding: 14px 18px;
            border-left: 4px solid #666;
            background: #f7f7f7;
            line-height: 1.55;
        ">
            <div>{title}</div>
            <div><em>{authors}</em></div>
            <div>{publication}</div>
            <div>{doi}</div>
        </div>
        """

        blocks.append(block)

    return "".join(blocks)


def on_format_clicked(_):
    global formatted_plain_text

    with output_area:
        output_area.clear_output()

        try:
            references = format_medline_text(input_box.value)
            formatted_plain_text = build_plain_text(references)

            display(HTML(
                f"""
                <h4>Formatted references</h4>
                {build_html(references)}
                """
            ))

            print("Plain-text version:\n")
            print(formatted_plain_text)

        except Exception as error:
            formatted_plain_text = ""
            print(f"Error: {error}")


def on_clear_clicked(_):
    global formatted_plain_text

    input_box.value = ""
    formatted_plain_text = ""

    with output_area:
        output_area.clear_output()


def on_copy_clicked(_):
    if not formatted_plain_text:
        with output_area:
            print("Format the references before copying.")
        return

    escaped_text = (
        formatted_plain_text
        .replace("\\", "\\\\")
        .replace("`", "\\`")
        .replace("${", "\\${")
    )

    display(HTML(f"""
    <script>
    navigator.clipboard.writeText(`{escaped_text}`);
    </script>
    <div style="color: green; margin-top: 8px;">
        Formatted references copied to the clipboard.
    </div>
    """))


format_button.on_click(on_format_clicked)
clear_button.on_click(on_clear_clicked)
copy_button.on_click(on_copy_clicked)

button_row = widgets.HBox([
    format_button,
    clear_button,
    copy_button
])

display(HTML("<h3>PubMed MEDLINE Reference Formatter</h3>"))
display(input_box)
display(button_row)
display(output_area)

Textarea(value='', layout=Layout(height='400px', width='100%'), placeholder='Paste one or more PubMed MEDLINE …

Output()